#### Word2Vec

- 문자를 수치형으로 변환시켜주는 딥러닝 기반의 임베딩 기술

<br>

- 매개변수
    - `sentences`
        - default : None
            - None이 default → 학습을 시킬 수 있다. 나중에 집어넣어도 된다는 뜻.
        - 토큰화가 된 문장 데이터 (2차원 데이터)
    - `vector_size`
        - default : 100
        - 임베딩 벡터 차원의 개수 (feature의 수)
    - `window`
        - default : 5
        - 예측 시 고려할 주변 단어와의 거리 (문맥의 크기)
    - `sg`
        - default : 0
            - 0인 경우
                - CBOW 방식 (주변 단어들을 이용하여 중심 단어를 예측)
            - 1인 경우
                - Skip_gram 방식 (중심 단어를 이용하여 주변 단어를 예측)
            - 빠른 계산이 필요한 경우라면 0, 일반적으로는 1을 사용
    - `min_count`
        - default : 5
        - 최소 등장 빈도 수
        - 적게 등장한 단어들을 제외
    - `hs`
        - default : 0
        - 계산 방식 지정
        - 0: Negative Sampling (계산량이 적음)
        - 1: Hierarchical Softmax (계산량 많음)
    - `epochs`
        - default : 100
        - 반복 학습 횟수 지정
    - `max_vocab_size`
        - default : None
        - 메모리 제한 시 사용할 최대 단어의 개수

<br>

- 속성
    - `wv`
        - 학습된 단어의 벡터값 (class 형태로 return)
        - 예시: model.wv['단어']
    - `wv.index_to_key`
        - 단어의 리스트 (학습된 단어의 개수) → 최소 등장 횟수에 영향
        - 등장 빈도 수에 따라 자동 정렬
    - `wv.key_to_index`
        - 단어 → 인덱스로 매칭
        - 특정 단어가 인덱스 몇에 위치하는가
    - `copus_total_word`
        - 전체 학습된 단어의 개수
    - `epochs`
        - 학습 epoch 수
    - `vector_size`
        - 벡터 차원의 수

<br>

- 메서드
    - `wv.most_similar(word, topn = 10)` 
        - 특정 단어와 유사한 단어를 출력
        - topn은 유사한 단어의 개수 지정
    - `wv.similarity(word1, word2)`
        - 두 단어 간의 코사인 유사도
    - `wv.get_vector(word)`
        - 특정 단어의 벡터를 반환
    - `train()`
        - 추가 데이터로 학습
    - `save()`
        - 학습된 모델을 저장
    - `Word2Vec.load()`
        - 저장되어 있는 모델을 로드

In [1]:
# !pip install gensim

In [2]:
from gensim.models import Word2Vec

In [3]:
from sklearn.svm import LinearSVC
from konlpy.tag import Komoran

In [4]:
docs = [
    '오늘 날씨가 좋다 여행 가고 싶다',
    '기온이 너무 올라서 아무것도 하기 싫다',
    '수업이 너무 지루하고 졸리다',
    '음식이 너무 맛이 없고 서비스도 별로다',
    '영화가 너무 재미있어서 시간이 가는 줄 몰랐다'
]

target = [1, 0, 0, 0, 1]

In [5]:
# Word2Vec은 토큰화된 데이터가 필요
komoran = Komoran()
allow_pos = ['NNP', 'NNG', 'VV', 'VA', 'SL', 'MAG']
tokens = []

for doc in docs:
    words = []
    for word, pos in komoran.pos(doc):
        if pos in allow_pos:
            words.append(word)
    tokens.append(words)

tokens

[['오늘', '날씨', '좋', '여행', '가'],
 ['기온', '너무', '오르', '아무것', '하', '싫'],
 ['수업', '너무', '졸리'],
 ['음식', '너무', '맛', '없', '서비스', '별로', '다'],
 ['영화', '너무', '재미있', '시간', '가', '모르']]

In [6]:
# Word2Vec을 이용하여 학습 (Skip-gram 방식)
w2v = Word2Vec(
    sentences = tokens,
    vector_size = 100,
    window = 5,
    min_count = 1,
    sg = 1,
    epochs = 100,
    seed = 42,
    workers = 2
)

In [7]:
# Word2Vec에서 wv 속성은 객체로 반환 → 자주 사용되는 객체임으로 변수에 저장
wv = w2v.wv

In [8]:
# wv에 특정 단어를 입력하면 벡터 출력
wv['여행']

array([ 0.00067393,  0.00060227,  0.0044239 , -0.00508487, -0.00337584,
       -0.00506212, -0.00414549, -0.00675128, -0.00933806,  0.00877077,
       -0.00856608,  0.00785082, -0.00989675,  0.00562794, -0.00337851,
        0.00042697, -0.00436719, -0.00029853, -0.00101309,  0.00058505,
       -0.00219302,  0.00076084,  0.00650614, -0.00127701, -0.00268797,
       -0.0073859 , -0.002808  , -0.00742177, -0.00639447,  0.00907987,
       -0.00822103, -0.00041097,  0.00665506,  0.00907863, -0.00704446,
       -0.00677937, -0.00250248,  0.00093587,  0.00914785, -0.00575276,
       -0.00664936, -0.00490768,  0.00760921, -0.00947477, -0.00244979,
       -0.00766411, -0.00686497,  0.00841504, -0.00982844, -0.00361847,
        0.00834756,  0.00207182, -0.0086334 , -0.000671  ,  0.00942244,
       -0.00202545,  0.00060924,  0.00068134,  0.00323355, -0.00620352,
        0.00690333,  0.00982491, -0.00727772,  0.00644986,  0.00644239,
        0.00483223, -0.00252508, -0.00059654,  0.00679188, -0.00

In [9]:
# 유사한 단어 찾기
wv.most_similar('날씨', topn = 3)

[('좋', 0.21210405230522156),
 ('기온', 0.1402115374803543),
 ('하', 0.12667463719844818)]

In [10]:
# 두 단어의 코사인 유사도를 확인
wv.similarity('여행', '음식')

np.float32(-0.07214948)

In [11]:
import numpy as np

In [12]:
# tokens의 단어들 중 w2v의 index_to_key에 존재하는 데이터의 단위 벡터를 확인

wv.index_to_key

['너무',
 '가',
 '모르',
 '시간',
 '재미있',
 '영화',
 '다',
 '별로',
 '서비스',
 '없',
 '맛',
 '음식',
 '졸리',
 '수업',
 '싫',
 '하',
 '아무것',
 '오르',
 '기온',
 '여행',
 '좋',
 '날씨',
 '오늘']

In [13]:
# from sklearn import set_config
# set_config(display = 'text')

In [21]:
# docs의 문장들을 벡터화한 리스트
vectors = []

for token in tokens:
    vec = []
    for word in token:
        # vec = []
        # 문제점? token의 각 원소를 word에 대입하여 반복 실행하면서 매번 초기화 → 마지막 단어의 벡터값만 vec에 대입
        if word in wv.index_to_key:
            # tokens 데이터에서 단어가 w2v의 학습 단어에 포함되어있을 때 해당 단어의 벡터 값을 vec에 추가
            vec.append(wv[word])
    print(np.array(vec).shape)
            
    vectors.append(np.mean(vec, axis = 0))

vectors

(5, 100)
(6, 100)
(3, 100)
(7, 100)
(6, 100)


[array([ 4.0627131e-03, -4.1177319e-03, -1.4225660e-03, -5.4318546e-03,
         2.9331266e-03, -1.0025300e-03, -5.0470594e-04, -2.1184415e-03,
        -2.8752084e-03,  6.8741088e-04, -2.7533637e-03,  4.1878768e-03,
        -9.9428801e-04,  2.8869177e-03, -5.6480215e-04, -1.9054037e-03,
        -1.4098274e-03,  1.4024605e-03, -1.1003042e-03, -2.0396772e-03,
         1.3082147e-03,  1.9095670e-03,  9.8853686e-04,  2.0640534e-04,
        -7.6428616e-05, -1.8237742e-03, -1.0836786e-03,  3.7147789e-03,
        -3.5496112e-03,  2.9948461e-03,  2.3482223e-04, -1.0175806e-03,
        -1.9324823e-04,  9.5323147e-04,  1.8641306e-03,  1.0443751e-03,
        -4.8639177e-04, -6.5589370e-03,  3.1723916e-03, -3.4027320e-04,
        -4.4211504e-04,  1.6320575e-03,  3.4491390e-03, -2.4417550e-03,
         8.0322695e-04,  2.9438485e-03, -4.4540949e-03,  1.3637433e-03,
         5.6419882e-04, -8.0012623e-04,  1.1474579e-03, -1.9369501e-03,
        -3.2696090e-04,  1.0688237e-03,  2.4639149e-03,  3.46896

In [15]:
np.array(vectors).shape

(5, 100)

In [16]:
from sklearn.svm import SVC

In [17]:
svc = SVC(random_state = 42)

In [18]:
# decode error 발생 시
# C:\Users\hkssn\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\_repr_html\estimator.py 오픈

svc.fit(vectors, target)

,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo random number generation for shuffling the data forprobability estimates. Ignored when `probability` is False.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide <shrinking_svm>`.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide <scores_probabilities>`...deprecated:: 1.9 The `probability` parameter is deprecated and will be removed in 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`.",'deprecated'
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None


In [19]:
svc.predict(vectors)

array([1, 0, 0, 0, 1])